# Instructblip Flan T5 Xl COCO Pruned GA P30 Mr002

This notebook was reorganized for the GitHub reproducibility package.
Original file: `GA-I_P30_MR0.02/InstructBLIP_Pruned[6,8,3]#L01f44c.ipynb`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


# **InstructBLIP Prune Çalışmaları**

In [ ]:
# toplam blok sayısı: 24

from transformers import InstructBlipForConditionalGeneration
import torch

model = InstructBlipForConditionalGeneration.from_pretrained(
    "Salesforce/instructblip-flan-t5-xl",
    torch_dtype=torch.float16
)

# Decoder yapısını görelim
decoder = model.language_model.decoder
print(type(decoder))
print(f"Blok sayısı: {len(decoder.block)}")
print(decoder.block[0])

In [ ]:
# toplam parametre sayısı

total = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {total:,}")
print(f"Toplam parametre (B): {total/1e9:.3f}B")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Not: BLIP-2 pruning çalışmasında COCO verisetinden elde ettiğimiz 200 adetlik proxy veri setini drive'a kaydetmiştik zaten, tekrardan veriseti hazırlamaya ve farklı seed kullanmaya gerek yok hem de doğru olmaz.

In [ ]:
!apt-get install -y default-jdk -q
!pip install pymoo pycocotools pycocoevalcap -q

import json, os, torch, time
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
from pycocoevalcap.cider.cider import Cider
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.core.callback import Callback
from google.colab import drive
drive.mount('/content/drive')

# ── Sabitler ─────────────────────────────────────────────────────────
MODEL_ID       = "Salesforce/instructblip-flan-t5-xl"
DEVICE         = "cuda"
BATCH_SIZE     = 8
N_BLOCKS       = 24
MAX_PRUNE      = 12

PROXY_DIR      = "/content/drive/MyDrive/proxy_coco_200"
PROXY_JSON     = os.path.join(PROXY_DIR, "proxy_coco_200_annotations.json")
PROXY_IMAGE    = os.path.join(PROXY_DIR, "images")
RESULTS_DIR    = "/content/drive/MyDrive/ga_results/instructblip"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Proxy veri seti ───────────────────────────────────────────────────
with open(PROXY_JSON, "r") as f:
    proxy_data = json.load(f)
proxy_images = proxy_data["images"]
print(f"✅ Proxy set yuklendi: {len(proxy_images)} goruntu")

# ── Model yükle ───────────────────────────────────────────────────────
print("Model yukleniyor...")
original_params = None
model     = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16).to(DEVICE).eval()
processor = InstructBlipProcessor.from_pretrained(MODEL_ID)
original_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model hazir | Parametre: {original_params:,}")

# ── Repair ───────────────────────────────────────────────────────────
def repair(chromosome, max_prune):
    chromosome = chromosome.copy()
    zero_indices = [i for i, bit in enumerate(chromosome) if bit == 0]
    if len(zero_indices) > max_prune:
        excess = len(zero_indices) - max_prune
        restore_indices = np.random.choice(zero_indices, excess, replace=False)
        for idx in restore_indices:
            chromosome[idx] = 1
    return chromosome

print("✅ repair hazir")

# ── Prune ────────────────────────────────────────────────────────────
import copy

def prune_model(model, chromosome):
    pruned_model = copy.deepcopy(model)
    # InstructBLIP T5 decoder katmanları
    blocks = pruned_model.language_model.decoder.block
    indices_to_keep = [i for i, bit in enumerate(chromosome) if bit == 1]
    pruned_model.language_model.decoder.block = torch.nn.ModuleList(
        [blocks[i] for i in indices_to_keep]
    )
    pruned_params  = sum(p.numel() for p in pruned_model.parameters())
    param_drop_rate = (original_params - pruned_params) / original_params
    return pruned_model, param_drop_rate

print("✅ prune_model hazir")

# ── Inference ────────────────────────────────────────────────────────
def run_inference(model, processor, proxy_images, device, batch_size=8):
    model.eval()
    results = {}
    for i in tqdm(range(0, len(proxy_images), batch_size), desc="Inference"):
        batch = proxy_images[i:i+batch_size]
        images, image_ids = [], []
        for item in batch:
            img_path = os.path.join(PROXY_IMAGE, item["filename"])
            images.append(Image.open(img_path).convert("RGB"))
            image_ids.append(item["image_id"])
        inputs = processor(
            images=images,
            text=[""] * len(images),
            return_tensors="pt"
        ).to(device, torch.float16)
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=50)
        captions = processor.batch_decode(generated_ids, skip_special_tokens=True)
        for img_id, caption in zip(image_ids, captions):
            results[str(img_id)] = [caption.strip()]
    return results

print("✅ run_inference hazir")

# ── CIDEr ────────────────────────────────────────────────────────────
def compute_cider(results, proxy_images):
    gts = {}
    for item in proxy_images:
        img_id = str(item["image_id"])
        gts[img_id] = [c.strip() for c in item["captions"]]
    scorer = Cider()
    score, _ = scorer.compute_score(gts, results)
    return score

print("✅ compute_cider hazir")

# ── Problem ──────────────────────────────────────────────────────────
class PruningProblem(Problem):
    def __init__(self):
        super().__init__(
            n_var=N_BLOCKS, n_obj=2, n_ieq_constr=0,
            xl=0, xu=1, vtype=bool
        )

    def _evaluate(self, X, out, *args, **kwargs):
        f1_list, f2_list = [], []
        for chromosome in X:
            chromosome          = repair(chromosome.astype(int), MAX_PRUNE)
            pruned_model, param_drop_rate = prune_model(model, chromosome)
            results             = run_inference(pruned_model, processor, proxy_images, DEVICE)
            cider_score         = compute_cider(results, proxy_images)
            f1_list.append(-param_drop_rate)
            f2_list.append(-cider_score)
            del pruned_model
            torch.cuda.empty_cache()
        out["F"] = np.column_stack([f1_list, f2_list])

# ── Callback ─────────────────────────────────────────────────────────
class EarlyStoppingCallback(Callback):
    def __init__(self, patience=20):
        super().__init__()
        self.patience   = patience
        self.no_improve = 0
        self.best_f     = None

    def notify(self, algorithm):
        current_f    = algorithm.pop.get("F")
        current_best = np.min(current_f[:, 1])

        if self.best_f is None or current_best < self.best_f:
            self.best_f     = current_best
            self.no_improve = 0
        else:
            self.no_improve += 1

        gen = algorithm.n_gen
        print(f"Nesil {gen:3d} | En iyi CIDEr: {-current_best:.4f} | Iyilesme yok: {self.no_improve}/{self.patience}")

        if self.no_improve >= self.patience:
            print("⛔ Erken durdurma tetiklendi!")
            algorithm.termination.force_termination = True

# ── GA ───────────────────────────────────────────────────────────────
algorithm = NSGA2(
    pop_size=30,
    sampling=BinaryRandomSampling(),
    crossover=SBX(prob=0.9),
    mutation=BitflipMutation(prob=0.02),
    eliminate_duplicates=True
)

termination = get_termination("n_gen", 50)

print("✅ GA kurulumu hazir")


In [ ]:
# ── Çalıştır ─────────────────────────────────────────────────────────
start_time = time.time()

print("🚀 GA basliyor...")
print(f"   Model    : {MODEL_ID}")
print(f"   Blok sayisi: {N_BLOCKS}")
print(f"   Populasyon: 30 | Nesil: 50 | Maks. budama: {MAX_PRUNE} blok")
print("-" * 60)

res = minimize(
    PruningProblem(),
    algorithm,
    termination,
    callback=EarlyStoppingCallback(patience=20),
    seed=42,
    verbose=False
)

elapsed = time.time() - start_time
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)

print("-" * 60)
print(f"✅ GA tamamlandi! Sure: {hours}s {minutes}dk {seconds}sn")
print(f"   Pareto cephesi: {len(res.F)} cozum")

# ── Pareto kaydet ─────────────────────────────────────────────────────
pareto_solutions = []
for i, (f, x) in enumerate(zip(res.F, res.X)):
    pareto_solutions.append({
        "index"          : i,
        "chromosome"     : x.tolist(),
        "param_drop_rate": float(-f[0]),
        "cider_score"    : float(-f[1])
    })

results_path = os.path.join(RESULTS_DIR, "pareto_solutions.json")
with open(results_path, "w") as f:
    json.dump({
        "model_id"       : MODEL_ID,
        "n_blocks"       : N_BLOCKS,
        "max_prune"      : MAX_PRUNE,
        "original_params": original_params,
        "solutions"      : pareto_solutions
    }, f, indent=2)

print(f"   Sonuclar kaydedildi: {results_path}")
print(f"\nPareto Cephesi:")
print(f"{'Cozum':>6} {'Param Dususu':>12} {'CIDEr':>8}")
print("-" * 35)
for sol in sorted(pareto_solutions, key=lambda x: x["param_drop_rate"]):
    print(f"{sol['index']:>6} %{sol['param_drop_rate']*100:>10.1f} {sol['cider_score']:>8.4f}")

In [ ]:
import json
import matplotlib.pyplot as plt

# Pareto çözümlerini oku
with open("/content/drive/MyDrive/ga_results/instructblip/pareto_solutions.json") as f:
    pareto_data = json.load(f)

solutions = sorted(pareto_data["solutions"], key=lambda x: x["param_drop_rate"])
pareto_x  = [s["param_drop_rate"] * 100 for s in solutions]
pareto_y  = [s["cider_score"] for s in solutions]

BASELINE      = 1.401
ORIGINAL_PARAMS = pareto_data["original_params"]

fig, ax1 = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor("#0f172a")
ax1.set_facecolor("#1e293b")
ax2 = ax1.twinx()

ax1.grid(color="#334155", linestyle="--", linewidth=0.6, alpha=0.7)

# Pareto cephesi
ax1.plot(pareto_x, pareto_y, color="#475569", linestyle="--", linewidth=1.2, zorder=2)
ax1.scatter(pareto_x, pareto_y, color="#38bdf8", s=80, zorder=3, label="Pareto noktaları (proxy, 200 görüntü)")

# Nokta etiketleri
for s in solutions:
    ax1.annotate(
        f"%{s['param_drop_rate']*100:.1f}\n{s['cider_score']:.4f}",
        (s["param_drop_rate"] * 100, s["cider_score"]),
        textcoords="offset points", xytext=(0, 12), ha="center",
        color="#38bdf8", fontsize=7.5,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="#0f172a", edgecolor="#38bdf8", alpha=0.8)
    )

# Baseline
ax1.axhline(y=BASELINE, color="#f59e0b", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.text(19.0, BASELINE + 0.01, f"Baseline: {BASELINE}", color="#f59e0b", fontsize=9, ha="right")

# Sağ eksen: Kalan parametre
remaining = [ORIGINAL_PARAMS * (1 - s["param_drop_rate"]) / 1e9 for s in solutions]
ax2.plot(pareto_x, remaining, color="#a78bfa", linestyle="-", linewidth=1.5, zorder=6, label="Kalan parametre (B)")
ax2.scatter(pareto_x, remaining, color="#a78bfa", s=45, zorder=7)

# Eksen ayarları — sol
ax1.set_xlim(-0.5, 21)
ax1.set_ylim(0, 1.55)
ax1.set_xlabel("Parametre Düşüşü (%)", color="#94a3b8", fontsize=12)
ax1.set_ylabel("CIDEr Skoru (Proxy)", color="#94a3b8", fontsize=12)
ax1.tick_params(colors="#94a3b8")
for spine in ax1.spines.values():
    spine.set_edgecolor("#334155")

# Eksen ayarları — sağ
ax2.set_ylabel("Kalan Parametre Sayısı (Milyar)", color="#a78bfa", fontsize=11)
ax2.tick_params(colors="#a78bfa")
for spine in ax2.spines.values():
    spine.set_edgecolor("#334155")
ax2.spines["right"].set_edgecolor("#a78bfa")

# Legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           facecolor="#1e293b", edgecolor="#334155", labelcolor="#94a3b8", fontsize=9, loc="upper right")

ax1.set_title("InstructBLIP — Pareto Cephesi (Proxy, 200 Görüntü)", color="#f1f5f9", fontsize=13, pad=14)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ga_results/instructblip/instructblip_pareto_proxy.png",
            dpi=150, bbox_inches="tight", facecolor="#0f172a")
plt.show()
print("✅ Kaydedildi")

✈ 6, 8 ve 3 numaralı indeksleri final evaluationa taşıyoruz.

# **Final Evaluation w/ COCO 5k Karpathy Test Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y default-jdk -q

import json, os, torch, time
from PIL import Image
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from google.colab import drive
drive.mount('/content/drive')

MODEL_ID    = "Salesforce/instructblip-flan-t5-xl"
DEVICE      = "cuda"
BATCH_SIZE  = 8
IMG_DIR     = "/content/drive/MyDrive/datasets/coco2014/val2014"
GT_JSON     = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json"
TEST_JSON   = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json"
PARETO_JSON = "/content/drive/MyDrive/ga_results/instructblip/pareto_solutions.json"
SAVE_DIR    = "/content/drive/MyDrive/ga_results/instructblip/final_eval"
os.makedirs(SAVE_DIR, exist_ok=True)

SELECTED  = [6, 8, 3]
label_map = {6: "low", 8: "mid", 3: "high"}

with open(TEST_JSON) as f:
    test_images = json.load(f)
print(f"Test seti: {len(test_images)} görüntü")

with open(PARETO_JSON) as f:
    all_solutions = json.load(f)
selected_solutions = [s for s in all_solutions["solutions"] if s["index"] in SELECTED]
selected_solutions.sort(key=lambda x: x["param_drop_rate"])
print("Seçilen çözümler:")
for s in selected_solutions:
    print(f"  index:{s['index']} | %{s['param_drop_rate']*100:.1f} | CIDEr:{s['cider_score']:.4f}")

def run_inference_full(model, processor, images, device, batch_size=8):
    results = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        pil_imgs, img_ids = [], []
        for img_info in batch:
            fname  = img_info["image"].split("/")[-1]
            img_id = int(fname.split("_")[-1].split(".")[0])
            pil_imgs.append(Image.open(os.path.join(IMG_DIR, fname)).convert("RGB"))
            img_ids.append(img_id)
        inputs = processor(
            images=pil_imgs,
            text=[""] * len(pil_imgs),
            return_tensors="pt"
        ).to(device, torch.float16)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=30)
        captions = processor.batch_decode(out, skip_special_tokens=True)
        for img_id, cap in zip(img_ids, captions):
            results.append({"image_id": img_id, "caption": cap.strip()})
        if (i // batch_size) % 50 == 0:
            print(f"  {i+len(batch)}/{len(images)}")
    return results

def compute_metrics(results):
    res_path = os.path.join(SAVE_DIR, "_tmp_results.json")
    with open(res_path, "w") as f:
        json.dump(results, f)
    coco_gt   = COCO(GT_JSON)
    coco_res  = coco_gt.loadRes(res_path)
    evaluator = COCOEvalCap(coco_gt, coco_res)
    try:
        evaluator.evaluate()
    except Exception as e:
        print(f"  ⚠️ SPICE hatası (görmezden gelindi): {e}")
    return {k: v for k, v in evaluator.eval.items()}

final_results = []

for sol in selected_solutions:
    label = label_map[sol["index"]]
    print(f"\n{'='*50}")
    print(f"[{label.upper()}] index:{sol['index']} | %{sol['param_drop_rate']*100:.1f} düşüş")

    print("Model yükleniyor...")
    model = InstructBlipForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16).to(DEVICE).eval()
    processor = InstructBlipProcessor.from_pretrained(MODEL_ID)

    chromosome   = sol["chromosome"]
    blocks       = model.language_model.decoder.block
    keep_indices = [i for i, keep in enumerate(chromosome) if keep]
    model.language_model.decoder.block = torch.nn.ModuleList(
        [blocks[i] for i in keep_indices])
    print(f"  {sum(1 for x in chromosome if not x)} blok silindi, {len(keep_indices)} kaldı")

    t0      = time.time()
    results = run_inference_full(model, processor, test_images, DEVICE, BATCH_SIZE)
    print(f"  Inference: {(time.time()-t0)/60:.1f} dk")

    metrics = compute_metrics(results)
    print(f"  CIDEr : {metrics.get('CIDEr', 0):.4f}")
    print(f"  BLEU-4: {metrics.get('Bleu_4', 0):.4f}")

    entry = {
        "label"          : label,
        "index"          : sol["index"],
        "param_drop_rate": sol["param_drop_rate"],
        "proxy_cider"    : sol["cider_score"],
        "final_cider"    : metrics.get("CIDEr", 0),
        "final_bleu4"    : metrics.get("Bleu_4", 0),
        "chromosome"     : chromosome
    }
    final_results.append(entry)
    with open(os.path.join(SAVE_DIR, f"instructblip_{label}_eval.json"), "w") as f:
        json.dump(entry, f, indent=2)
    print(f"  ✅ Kaydedildi")

    del model
    torch.cuda.empty_cache()

summary_path = os.path.join(SAVE_DIR, "instructblip_final_summary.json")
with open(summary_path, "w") as f:
    json.dump(final_results, f, indent=2)

print(f"\n{'='*50}\nÖZET:")
print(f"{'Label':<8} {'Param Düşüş':>12} {'Proxy CIDEr':>12} {'Final CIDEr':>12} {'BLEU-4':>8}")
print("-"*55)
for r in final_results:
    print(f"{r['label']:<8} %{r['param_drop_rate']*100:>10.1f} {r['proxy_cider']:>12.4f} {r['final_cider']:>12.4f} {r['final_bleu4']:>8.4f}")

# **Final Evaluation w/ NoCaps 4.5k Val Test Dataset**

In [ ]:
!apt-get install -y default-jdk -q

import json, os, torch, time
from PIL import Image
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from google.colab import drive
drive.mount('/content/drive')

MODEL_ID    = "Salesforce/instructblip-flan-t5-xl"
DEVICE      = "cuda"
BATCH_SIZE  = 8
IMG_DIR     = "/content/drive/MyDrive/datasets/nocaps/images_val_hf"
GT_JSON     = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
PARETO_JSON = "/content/drive/MyDrive/ga_results/instructblip/pareto_solutions.json"
SAVE_DIR    = "/content/drive/MyDrive/ga_results/instructblip/final_eval_nocaps"
os.makedirs(SAVE_DIR, exist_ok=True)

SELECTED  = [6, 8, 3]
label_map = {6: "low", 8: "mid", 3: "high"}

with open(GT_JSON) as f:
    nocaps_data = json.load(f)
test_images = nocaps_data["images"]
print(f"NoCaps val seti: {len(test_images)} görüntü")

with open(PARETO_JSON) as f:
    all_solutions = json.load(f)
selected_solutions = [s for s in all_solutions["solutions"] if s["index"] in SELECTED]
selected_solutions.sort(key=lambda x: x["param_drop_rate"])
print("Seçilen çözümler:")
for s in selected_solutions:
    print(f"  index:{s['index']} | %{s['param_drop_rate']*100:.1f} | CIDEr:{s['cider_score']:.4f}")

def run_inference_full(model, processor, images, device, batch_size=8):
    results = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        pil_imgs, img_ids = [], []
        for img_info in batch:
            fname  = img_info["file_name"]
            img_id = img_info["id"]
            pil_imgs.append(Image.open(os.path.join(IMG_DIR, fname)).convert("RGB"))
            img_ids.append(img_id)
        inputs = processor(
            images=pil_imgs,
            text=[""] * len(pil_imgs),
            return_tensors="pt"
        ).to(device, torch.float16)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=30)
        captions = processor.batch_decode(out, skip_special_tokens=True)
        for img_id, cap in zip(img_ids, captions):
            results.append({"image_id": img_id, "caption": cap.strip()})
        if (i // batch_size) % 50 == 0:
            print(f"  {i+len(batch)}/{len(images)}")
    return results

def compute_metrics(results):
    res_path = os.path.join(SAVE_DIR, "_tmp_results.json")
    with open(res_path, "w") as f:
        json.dump(results, f)
    coco_gt   = COCO(GT_JSON)
    coco_res  = coco_gt.loadRes(res_path)
    evaluator = COCOEvalCap(coco_gt, coco_res)
    try:
        evaluator.evaluate()
    except Exception as e:
        print(f"  ⚠️ SPICE hatası (görmezden gelindi): {e}")
    return {k: v for k, v in evaluator.eval.items()}

final_results = []

for sol in selected_solutions:
    label = label_map[sol["index"]]
    print(f"\n{'='*50}")
    print(f"[{label.upper()}] index:{sol['index']} | %{sol['param_drop_rate']*100:.1f} düşüş")

    print("Model yükleniyor...")
    model = InstructBlipForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16).to(DEVICE).eval()
    processor = InstructBlipProcessor.from_pretrained(MODEL_ID)

    chromosome   = sol["chromosome"]
    blocks       = model.language_model.decoder.block
    keep_indices = [i for i, keep in enumerate(chromosome) if keep]
    model.language_model.decoder.block = torch.nn.ModuleList(
        [blocks[i] for i in keep_indices])
    print(f"  {sum(1 for x in chromosome if not x)} blok silindi, {len(keep_indices)} kaldı")

    t0      = time.time()
    results = run_inference_full(model, processor, test_images, DEVICE, BATCH_SIZE)
    print(f"  Inference: {(time.time()-t0)/60:.1f} dk")

    metrics = compute_metrics(results)
    print(f"  CIDEr : {metrics.get('CIDEr', 0):.4f}")
    print(f"  BLEU-4: {metrics.get('Bleu_4', 0):.4f}")

    entry = {
        "label"          : label,
        "index"          : sol["index"],
        "param_drop_rate": sol["param_drop_rate"],
        "proxy_cider"    : sol["cider_score"],
        "final_cider"    : metrics.get("CIDEr", 0),
        "final_bleu4"    : metrics.get("Bleu_4", 0),
        "chromosome"     : chromosome
    }
    final_results.append(entry)
    with open(os.path.join(SAVE_DIR, f"instructblip_{label}_nocaps.json"), "w") as f:
        json.dump(entry, f, indent=2)
    print(f"  ✅ Kaydedildi")

    del model
    torch.cuda.empty_cache()

summary_path = os.path.join(SAVE_DIR, "instructblip_nocaps_summary.json")
with open(summary_path, "w") as f:
    json.dump(final_results, f, indent=2)

print(f"\n{'='*50}\nÖZET:")
print(f"{'Label':<8} {'Param Düşüş':>12} {'Proxy CIDEr':>12} {'Final CIDEr':>12} {'BLEU-4':>8}")
print("-"*55)
for r in final_results:
    print(f"{r['label']:<8} %{r['param_drop_rate']*100:>10.1f} {r['proxy_cider']:>12.4f} {r['final_cider']:>12.4f} {r['final_bleu4']:>8.4f}")